In [ ]:
import re
import sys
import time
import random

import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

UA = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ru-RU,ru;q=0.9",
    "Accept": "text/html,application/xhtml+xml,*/*;q=0.8",
}


def rub(x, _=None):
    return f"{int(x):,}".replace(",", "\u202f")


def fetch(url):
    time.sleep(random.uniform(1.0, 2.0))
    s = requests.Session()
    s.headers.update(UA)
    r = s.get(url, timeout=15)
    if r.status_code != 200:
        print(f"! {r.status_code} на {url}")
        return None
    return BeautifulSoup(r.text, "html.parser")


def clean_price(text):
    digits = re.sub(r"[^\d]", "", str(text))
    if digits and 300 <= int(digits) <= 100_000:
        return int(digits)
    return None


def extract_count(text):
    m = re.search(r"(\d+)", str(text))
    if m:
        val = int(m.group(1))
        if 1 <= val <= 999:
            return val
    return None


def normalize_to_15(price, count):
    if not count or count == 0:
        return price
    return round(price / count * 15)


def make_label(text, count):
    if count and count != 15:
        return f"{text[:50]} [×15]"
    return text[:60]


class FlowwowScraper:

    main_file  = "Монобукет_ купить классические букеты с доставкой в Москве _ Flowwow.htm"
    extra_files = [
        "Монобукеты_герберы.htm",
        "Монобукеты_хризантемы.htm",
    ]

    def parse_file(self, filepath):
        try:
            with open(filepath, encoding="utf-8") as f:
                html = f.read()
        except FileNotFoundError:
            return []

        soup = BeautifulSoup(html, "html.parser")
        rows = []
        for card in soup.select("article.main-block-content-products-item"):
            name_el  = card.select_one("div.name")
            price_el = card.select_one("div.price span")
            name  = name_el.get_text(strip=True) if name_el else ""
            price = clean_price(price_el.get_text() if price_el else "")
            if not name or not price:
                continue
            count = extract_count(name)
            if not count:
                continue
            rows.append({"name": make_label(name, count), "price": normalize_to_15(price, count)})
        return rows

    def run(self):
        result = self.parse_file(self.main_file)
        for filepath in self.extra_files:
            result.extend(self.parse_file(filepath))
        return list({(r["name"], r["price"]): r for r in result}.values())


class DostavkaScraper:

    base_url = "https://www.dostavka-tsvetov.com/tsvety/monobuket"

    def run(self):
        result = []
        for page in range(1, 6):
            url  = self.base_url if page == 1 else f"{self.base_url}?page={page}"
            soup = fetch(url)
            if not soup:
                break

            cards = soup.select("div.item_box")
            if not cards:
                break

            for card in cards:
                name_el  = card.select_one("div.item-name")
                price_el = card.select_one("span.price")
                name  = name_el.get_text(strip=True) if name_el else ""
                price = clean_price(price_el.get_text() if price_el else "")
                if not name or not price:
                    continue
                count = extract_count(name)
                if not count:
                    continue
                result.append({"name": make_label(name, count), "price": normalize_to_15(price, count)})

            if not soup.select_one("a[rel='next'], a[class*='next'], .pagination-next"):
                break

        return list({(r["name"], r["price"]): r for r in result}.values())


class UflorScraper:

    base_url = "https://uflor.ru/catalog/monobukety/"

    def run(self):
        result = []
        for page in range(1, 6):
            url  = self.base_url if page == 1 else f"{self.base_url}?page={page}"
            soup = fetch(url)
            if not soup:
                break

            cards = soup.select("div.product-item")
            if not cards:
                break

            for card in cards:
                desc_el  = card.select_one("div.product-item__description")
                price_el = card.select_one("div.product-item__price")
                desc  = desc_el.get_text(strip=True) if desc_el else ""
                price = clean_price(price_el.get_text() if price_el else "")
                if not desc or not price:
                    continue
                count = extract_count(desc)
                if not count:
                    continue
                result.append({"name": make_label(desc, count), "price": normalize_to_15(price, count)})

            if not soup.select_one("a[rel='next'], [class*='pagination__next'], [class*='next-page']"):
                break

        return list({(r["name"], r["price"]): r for r in result}.values())


class ShopChart:

    def __init__(self, rows, title, color, fname):
        self.rows  = rows
        self.title = title
        self.color = color
        self.fname = fname

    def draw(self):
        df    = pd.DataFrame(self.rows).sort_values("price").reset_index(drop=True)
        n     = len(df)
        mean_v = df["price"].mean()
        med_v  = df["price"].median()
        min_v  = df["price"].min()
        max_v  = df["price"].max()

        fig, (left, right) = plt.subplots(
            1, 2, figsize=(18, max(7, n * 0.38)),
            gridspec_kw={"width_ratios": [3, 1]},
        )
        fig.suptitle(self.title, fontsize=14, fontweight="bold", y=1.01)

        bars = left.barh(df["name"], df["price"],
                         color=self.color, edgecolor="white", height=0.72)
        left.set_xlabel("Цена за 15 цветков, руб.", fontsize=11)
        left.set_title("Название и цена (пересчитано на 15 шт.)", fontsize=12, fontweight="bold")
        left.xaxis.set_major_formatter(mticker.FuncFormatter(rub))
        left.grid(axis="x", alpha=0.25)
        left.set_axisbelow(True)
        left.tick_params(axis="y", labelsize=8)

        gap = max_v * 0.008
        for bar, val in zip(bars, df["price"]):
            left.text(val + gap, bar.get_y() + bar.get_height() / 2,
                      f"{val:,}\u202f₽".replace(",", "\u202f"),
                      va="center", fontsize=7.5)

        left.axvline(mean_v, color="#E76F51", lw=1.6, ls="--",
                     label=f"Среднее\u2002{mean_v:,.0f}\u202f₽".replace(",", "\u202f"))
        left.axvline(med_v, color="#264653", lw=1.6, ls=":",
                     label=f"Медиана\u2002{med_v:,.0f}\u202f₽".replace(",", "\u202f"))
        left.legend(fontsize=9, loc="lower right")

        stat_bars = right.bar(["Среднее", "Медиана"], [mean_v, med_v],
                              color=["#E76F51", "#264653"], edgecolor="white", width=0.45)
        right.set_ylabel("Цена, руб.", fontsize=11)
        right.set_title("Среднее и медиана\n(за 15 цветков)", fontsize=12, fontweight="bold")
        right.yaxis.set_major_formatter(mticker.FuncFormatter(rub))
        right.grid(axis="y", alpha=0.25)
        right.set_axisbelow(True)

        top = max_v * 0.025
        for bar, val in zip(stat_bars, [mean_v, med_v]):
            right.text(bar.get_x() + bar.get_width() / 2, val + top,
                       f"{val:,.0f}\u202f₽".replace(",", "\u202f"),
                       ha="center", fontsize=12, fontweight="bold")

        note = (f"Позиций:  {n}\n"
                f"Мин:  {min_v:,}\u202f₽\n".replace(",", "\u202f") +
                f"Макс: {max_v:,}\u202f₽".replace(",", "\u202f"))
        right.text(0.5, 0.06, note, transform=right.transAxes,
                   ha="center", va="bottom", fontsize=9,
                   bbox=dict(boxstyle="round,pad=0.5", facecolor="#f0f9f4", edgecolor=self.color))

        plt.tight_layout()
        plt.savefig(self.fname, dpi=150, bbox_inches="tight")
        plt.close()


class ComparisonChart:

    def __init__(self, shops):
        self.shops = [s for s in shops if s["rows"]]

    def draw(self):
        if not self.shops:
            return

        labels  = [s["label"] for s in self.shops]
        colors  = [s["color"] for s in self.shops]
        data    = [pd.Series([r["price"] for r in s["rows"]]) for s in self.shops]
        means   = [d.mean()   for d in data]
        medians = [d.median() for d in data]
        mins    = [d.min()    for d in data]
        maxs    = [d.max()    for d in data]
        stds    = [d.std()    for d in data]
        counts  = [len(d)     for d in data]

        fig = plt.figure(figsize=(18, 14))
        fig.suptitle("Сравнение цен на монобукеты (пересчитано на 15 цветков) — маркетплейсы",
                     fontsize=14, fontweight="bold", y=1.01)
        gs = fig.add_gridspec(2, 2, hspace=0.45, wspace=0.35)

        ax = fig.add_subplot(gs[0, 0])
        bars = ax.bar(labels, means, color=colors, edgecolor="white", width=0.5)
        ax.set_title("Средняя цена за 15 цветков", fontsize=12, fontweight="bold")
        ax.set_ylabel("Руб.")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(rub))
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)
        for bar, val in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width() / 2, val + max(means) * 0.02,
                    f"{val:,.0f}\u202f₽".replace(",", "\u202f"),
                    ha="center", fontsize=11, fontweight="bold")

        ax = fig.add_subplot(gs[0, 1])
        bars = ax.bar(labels, medians, color=colors, edgecolor="white", width=0.5)
        ax.set_title("Медианная цена за 15 цветков", fontsize=12, fontweight="bold")
        ax.set_ylabel("Руб.")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(rub))
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)
        for bar, val in zip(bars, medians):
            ax.text(bar.get_x() + bar.get_width() / 2, val + max(medians) * 0.02,
                    f"{val:,.0f}\u202f₽".replace(",", "\u202f"),
                    ha="center", fontsize=11, fontweight="bold")

        ax = fig.add_subplot(gs[1, 0])
        bp = ax.boxplot([d.tolist() for d in data], tick_labels=labels,
                        patch_artist=True, widths=0.45,
                        medianprops=dict(color="white", linewidth=2))
        for patch, c in zip(bp["boxes"], colors):
            patch.set_facecolor(c)
            patch.set_alpha(0.82)
        for elem in ["whiskers", "caps", "fliers"]:
            for line in bp[elem]:
                line.set_color("#555")
        ax.set_title("Разброс цен (за 15 цветков)", fontsize=12, fontweight="bold")
        ax.set_ylabel("Руб.")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(rub))
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)

        ax = fig.add_subplot(gs[1, 1])
        ax.axis("off")
        ax.set_title("Ключевые метрики (15 цветков)", fontsize=12, fontweight="bold")

        headers = ["Магазин", "Поз.", "Мин ₽", "Среднее ₽", "Медиана ₽", "Макс ₽", "Разброс ₽"]
        rows = []
        for i in range(len(self.shops)):
            rows.append([
                labels[i],
                str(counts[i]),
                f"{int(mins[i]):,}".replace(",", "\u202f"),
                f"{int(means[i]):,}".replace(",", "\u202f"),
                f"{int(medians[i]):,}".replace(",", "\u202f"),
                f"{int(maxs[i]):,}".replace(",", "\u202f"),
                f"{int(stds[i]):,}".replace(",", "\u202f"),
            ])

        tbl = ax.table(cellText=rows, colLabels=headers,
                       cellLoc="center", loc="upper center",
                       bbox=[0, 0.38, 1, 0.58])
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(8.5)
        for j in range(len(headers)):
            tbl[0, j].set_facecolor("#2D6A4F")
            tbl[0, j].set_text_props(color="white", fontweight="bold")
        for i in range(1, len(rows) + 1):
            bg = "#f0f9f4" if i % 2 == 0 else "white"
            for j in range(len(headers)):
                tbl[i, j].set_facecolor(bg)

        cheapest = labels[means.index(min(means))]
        widest   = labels[stds.index(max(stds))]

        notes = [
            f"Самая низкая средняя цена: {cheapest}",
            f"Наибольший разброс: {widest}",
            "(широкий разброс = широкий ассортимент по цене)",
            "",
            "Вывод:",
            f"Ориентир для ценообразования — медиана {labels[medians.index(min(medians))]}",
            "Выше медианы = премиум-позиционирование",
            "Ниже = конкуренция по цене",
        ]
        y = 0.33
        for line in notes:
            bold  = "bold"    if line.startswith("Вывод") else "normal"
            color = "#1B4332" if line.startswith("Вывод") else "#333"
            ax.text(0.02, y, line, transform=ax.transAxes,
                    fontsize=8.5, va="top", color=color, fontweight=bold)
            y -= 0.075

        plt.savefig("04_marketplaces_comparison.png", dpi=150, bbox_inches="tight")
        plt.close()


def main():
    flowwow  = FlowwowScraper().run()
    dostavka = DostavkaScraper().run()
    uflor    = UflorScraper().run()

    if not any([flowwow, dostavka, uflor]):
        print("Нет данных ни с одного источника")
        sys.exit(1)

    all_data = (
        [{"source": "Flowwow",  **r} for r in flowwow]  +
        [{"source": "Dostavka", **r} for r in dostavka] +
        [{"source": "Uflor",    **r} for r in uflor]
    )
    pd.DataFrame(all_data).to_csv("marketplaces_data.csv", index=False, encoding="utf-8-sig")

    if flowwow:
        ShopChart(flowwow,  "Flowwow — монобукеты (цена за 15 цветков)",          "#E76F51", "01_flowwow.png").draw()
    if dostavka:
        ShopChart(dostavka, "Dostavka-tsvetov — монобукеты (цена за 15 цветков)", "#2D6A4F", "02_dostavka.png").draw()
    if uflor:
        ShopChart(uflor,    "Uflor — монобукеты (цена за 15 цветков)",            "#74C69D", "03_uflor.png").draw()

    ComparisonChart([
        {"label": "Flowwow",  "rows": flowwow,  "color": "#E76F51"},
        {"label": "Dostavka", "rows": dostavka, "color": "#2D6A4F"},
        {"label": "Uflor",    "rows": uflor,    "color": "#74C69D"},
    ]).draw()

    print("Готово! Созданные файлы:")
    print("  marketplaces_data.csv")
    if flowwow:  print("  01_flowwow.png")
    if dostavka: print("  02_dostavka.png")
    if uflor:    print("  03_uflor.png")
    print("  04_marketplaces_comparison.png")


if __name__ == "__main__":
    main()

Готово! Созданные файлы:
  marketplaces_data.csv
  02_dostavka.png
  03_uflor.png
  04_marketplaces_comparison.png
